# 多核 SPMD 与 SIMD/SIMT 并行范式

本节建立三个贯穿全课程的并行概念：**SPMD**（多核怎么分工）、**SIMD**（核内怎么批量算）、**SIMT**（核内怎么逐线程算）。学完本节，你将拥有一张"范式选型"的直觉地图，并能读懂后续章节中所有并行相关的表述。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 两个层面的并行

PyPTO Pro 的并行执行发生在两个层面：

- **多核层**：多个 AI Core 以 **SPMD** 方式并行，各自处理不同的数据分片；
- **核内层**：每个 AI Core 内部，用 **SIMD**（批量数据）或 **SIMT**（多线程）组织计算。

两层是叠加关系：先用 SPMD 把数据切开分给各核，每个核再用 SIMD 或 SIMT 处理自己的分片。


## 2. SPMD：一份程序，多核分片

SPMD（Single Program Multiple Data，单程序多数据）：多个核执行**同一份 Kernel 程序**，各核根据自身**核索引**处理不同的数据分片。

Host 侧通过启动配置 `block_dim` 指定核数；Kernel 内可用两个接口感知多核环境（用法详见第 2 章，此处仅建立概念）：

```python
import pypto_pro.language as pl

num_cores = pl.get_block_num()   # 实际生效核数
core_id = pl.get_block_idx()     # 当前执行域的核索引

# 典型分片模式：第 core_id 个核以 num_cores 为步长处理数据块
for tile_idx in pl.range(core_id, tile_num, num_cores):
    ...
```

【待 950 验证：以上代码为概念示意，取自官方《SIMD 编程范式》文档示例，未在本机执行】

SPMD 的要点：**不同核之间不保证执行顺序**，各自独立推进自己的分片——这也是第 2 章"核间同步"存在的原因。


## 3. SIMD：一条指令管一整块

SIMD（Single Instruction Multiple Data，单指令多数据）：**一条指令对多个数据通道施加相同的操作**。指令延迟可能跨多个周期，一个 Tile API 也可能展开为多条指令，不能把一次 API 调用等同于一个时钟周期。

你其实早就熟悉这种思维方式——NumPy 里 `c = a + b` 作用于整个数组、PyTorch 里张量运算天然广播，都是"对一整块数据做同一件事"的表达。Pro 的 SIMD 编程把这个直觉带到硬件层面：以 **Tile**（UB 中的数据块）或 **Vector Register**（寄存器）为主要编程对象，一次接口调用处理整个数据块。

SIMD 的核心特征与适用场景：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">特征</th><th style="text-align:left;">含义</th></tr>
<tr><td>单指令驱动</td><td>所有并行计算单元同步执行同一条指令，操作完全一致</td></tr>
<tr><td>数据同构</td><td>要求参与计算的数据类型统一、长度相同</td></tr>
<tr><td>同步执行</td><td>参与同一条 SIMD 指令的数据通道执行相同操作；整个 Tile 的处理时长取决于大小、指令展开和流水</td></tr>
</table>

因此 SIMD 适合：**连续规整的数据访问**、**对大量数据执行相同操作**的场景——逐元素计算、归约、矩阵乘、融合计算等核心计算路径。


## 4. SIMT：一个线程管一个元素

SIMT（Single Instruction Multiple Threads，单指令多线程）：**同一条指令由多个独立线程并行执行，每个线程通过自身索引处理不同的数据**。

如果你写过 Python 的循环处理每个元素，那你已经拥有 SIMT 的心智模型：

```python
# 你熟悉的逐元素直觉
for i in range(n):
    c[i] = a[i] + b[i]    # 第 i 次"迭代"只关心第 i 个元素
```

SIMT 把这个直觉搬到硬件上：每个线程有自己的索引、独立的执行状态，**可以独立寻址、进入不同分支**。这与 SIMD"所有元素同步走同一条路"形成鲜明对比。

SIMT 的核心特征与适用场景：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">特征</th><th style="text-align:left;">含义</th></tr>
<tr><td>多线程并行</td><td>同一份程序由多个线程并行执行，每线程处理不同数据</td></tr>
<tr><td>线程独立</td><td>线程拥有独立索引与执行状态，可独立寻址、支持分支与循环</td></tr>
<tr><td>并行调度</td><td>线程由硬件并行调度，不同线程可走不同控制路径</td></tr>
</table>

因此 SIMT 适合：**离散索引与不规则数据访问**、**分支较多的处理逻辑**、**直方图/计数器等原子更新**、**稀疏与动态依赖较强的局部计算**。


## 5. 对比与选型直觉

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">维度</th><th style="text-align:left;">SIMD</th><th style="text-align:left;">SIMT</th></tr>
<tr><td>直觉来源</td><td>NumPy/PyTorch 整张量运算</td><td>Python 逐元素循环</td></tr>
<tr><td>编程对象</td><td>Tile / Vector Register</td><td>线程（线程索引寻址）</td></tr>
<tr><td>控制流</td><td>整体同步，无独立分支</td><td>线程独立分支、循环</td></tr>
<tr><td>数据形态</td><td>规整、同构、连续</td><td>不规则、离散、动态</td></tr>
<tr><td>典型算子</td><td>elementwise、matmul、softmax、归一化</td><td>embedding 索引、直方图、稀疏选择</td></tr>
</table>

选型一句话：**数据规整、操作一致 → SIMD；访问不规则、逻辑逐个判断 → SIMT**。更常见的情况是两者出现在同一个算子里——稠密部分走 SIMD，稀疏/不规则部分走 SIMT，这正是第 9 章"协同编程"的主题。


## 6. 组合形态预告

本课程主线使用 **SIMD Kernel 内调用 SIMT 函数**的混合方式：外层 JIT Kernel 组织 Tile、搬运和启动，`@pl.vector_function(mode="simt", max_threads=...)` 描述逐线程逻辑。当前 PyPTO 源码还提供 `@pl.jit(mode="simt", max_threads=...)` 的 Host 直接启动入口，具体发行包支持情况须核对版本；本课程不将该入口作为运行前提。下一节先对照学习混合方式。


### 用 8 行数据区分两层索引

假设将 [8,256] 按行分给 2 个核：核 0 处理行 0～3，核 1 处理行 4～7。核内若使用 256 个 SIMT 线程，线程 t 处理自己分片中列 t 的四个元素。核编号选择数据分片，线程编号选择分片内部的工作；二者都不能省略。

同一任务也能用 SIMD 对 [4,256] Tile 做一次 `pl.add`。这不意味着一条硬件指令完成 1024 次加法，只意味着用一个块级接口表达这段计算。两种写法必须覆盖相同的全局坐标集合，才有比较正确性与性能的前提。


## 7. 课后练习

1. （选择）"每个线程可以通过自身索引进入不同分支"，描述的是：
   A. SPMD<br/>B. SIMD<br/>C. SIMT
2. （填空）多核层面，各核执行同一份程序、按核索引处理不同分片，这种并行方式称为 ______；Host 侧通过 ______ 配置核数。
3. （选择）实现 Embedding 查表（按索引离散取数）时，更自然的范式是：
   A. SIMD<br/>B. SIMT
4. （判断）本节采用的 `@pl.vector_function(mode="simt", max_threads=...)` 函数，可在 Host 上直接按普通 Python 函数调用来启动 NPU 计算。（对/错）

In [ ]:
!cat ./answer/01.05_answer.txt

## 8. 本节小结

- 两层并行：**多核 SPMD 分片 + 核内 SIMD/SIMT 计算**；
- SIMD = 一条指令管一整块（NumPy/PyTorch 直觉），适合规整数据；
- SIMT = 一个线程管一个元素（逐元素循环直觉），适合不规则访问与分支逻辑；
- 本课程采用 **SIMT 函数嵌于外层 SIMD Kernel** 的混合方式，两者在同一算子中协同。

下一节：动手时间——搭建环境，跑通你的第一个 PyPTO Pro Kernel。